# Multilayer Perceptrons & Activation Functions

CSCI 6379 · Topic 14. How a hidden layer solves XOR (it re-maps the inputs), why non-linearity is essential (linear layers collapse), and the common activation functions.

## Without an activation, stacked linear layers collapse into one

Two linear layers W2(W1 x + b1) + b2 = (W2 W1) x + (W2 b1 + b2) — a single linear layer. Verify numerically:

In [ ]:
import torch

torch.manual_seed(1)
W1, b1 = torch.randn(3, 2), torch.randn(3)
W2, b2 = torch.randn(1, 3), torch.randn(1)
x = torch.randn(5, 2)

stacked = (x @ W1.T + b1) @ W2.T + b2      # two linear layers, no activation
W_eq, b_eq = W2 @ W1, W2 @ b1 + b2         # collapsed to one layer
single = x @ W_eq.T + b_eq

print("max difference:", (stacked - single).abs().max().item())   # ~0 -> identical

## The hidden layer re-maps XOR so it becomes linearly separable

Train a 2-2-1 MLP on XOR, then read the two hidden-neuron activations for each input. In that hidden space the two '1' points collapse together and a single line separates the classes.

In [ ]:
import torch.nn as nn, torch.optim as optim

X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

torch.manual_seed(53)
fc1, fc2 = nn.Linear(2, 2), nn.Linear(2, 1)
opt = optim.Adam(list(fc1.parameters()) + list(fc2.parameters()), lr=0.1)
crit = nn.MSELoss()
for _ in range(3000):
    opt.zero_grad()
    loss = crit(torch.sigmoid(fc2(torch.sigmoid(fc1(X)))), y)
    loss.backward(); opt.step()

H = torch.sigmoid(fc1(X)).detach()
print("final loss:", round(loss.item(), 4))
for (a, b), lab, h in zip(X.tolist(), y.flatten().tolist(), H.tolist()):
    print(f"  input {a,b} label {int(lab)} -> hidden ({h[0]:.3f}, {h[1]:.3f})")

## The activation functions and their derivatives

Sigmoid and tanh saturate (their derivatives shrink toward 0), which causes the vanishing gradient problem in deep nets. ReLU's derivative is 0 or 1, so gradients do not shrink.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-6, 6, 400)
sig = 1/(1+np.exp(-z)); tanh = np.tanh(z)
relu = np.maximum(0, z); lrelu = np.where(z > 0, z, 0.1*z)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for f, name in [(sig,"sigmoid"), (tanh,"tanh"), (relu,"ReLU"), (lrelu,"Leaky ReLU")]:
    ax[0].plot(z, f, label=name)
ax[0].set_title("activations"); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_ylim(-1.5, 3)

ax[1].plot(z, sig*(1-sig), label="sigmoid' (max 0.25)")
ax[1].plot(z, 1-tanh**2, label="tanh' (max 1.0)")
ax[1].plot(z, (z > 0).astype(float), label="ReLU' (0 or 1)")
ax[1].set_title("derivatives"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# The vanishing gradient: multiplying the max derivative across N layers
for n in [1, 3, 5, 8]:
    print(f"{n} layers -> sigmoid factor {0.25**n:.2e},  ReLU factor {1.0**n:.0f}")